In [ ]:
!nvidia-smi

In [ ]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr evaluate -q

In [ ]:
!pip install "transformers<5.0.0"

In [ ]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
from datasets import load_dataset
import pandas as pd
!pip install evaluate
import evaluate
rouge = evaluate.load("rouge")

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_ckpt = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_bart = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

In [ ]:
dataset_dailymail = load_dataset("abisee/cnn_dailymail", "2.0.0")

In [ ]:
dataset_dailymail

In [ ]:
dataset_dailymail["train"]["article"][1]

In [ ]:
dataset_dailymail["train"]["highlights"][1]

In [ ]:
split_lengths = [len(dataset_dailymail[split])for split in dataset_dailymail]

print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_dailymail['train'].column_names}")
print("\narticle:")

print(dataset_dailymail["test"][1]["article"])

print("\nhighlights:")

print(dataset_dailymail["test"][1]["highlights"])

In [ ]:
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(example_batch['article'] , max_length = 1024, truncation = True )

    with tokenizer.as_target_tokenizer():
        target_encodings = tokenizer(example_batch['highlights'], max_length = 128, truncation = True )

    return {
        'input_ids' : input_encodings['input_ids'],
        'attention_mask': input_encodings['attention_mask'],
        'labels': target_encodings['input_ids']
    }

In [ ]:
dataset_dailymail_pt = dataset_dailymail.map(convert_examples_to_features, batched = True)

In [ ]:
dataset_dailymail_pt["train"]

In [ ]:
dataset_dailymail_pt["train"]["input_ids"][1]

In [ ]:
dataset_dailymail_pt["train"]["attention_mask"][1]

In [ ]:
# Training

from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_bart)

In [ ]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir='bart-dailymail',         # where to save checkpoints
    num_train_epochs=1,                  # number of training epochs
    warmup_steps=500,                    # learning rate warmup
    per_device_train_batch_size=1,       # batch size per device
    per_device_eval_batch_size=1,        # evaluation batch size
    weight_decay=0.01,                   # weight decay for optimizer
    gradient_accumulation_steps=16,      # effective batch size = 16 * 1
    logging_steps=200,                   # logs every 200 steps (less spam)
    evaluation_strategy="epoch",         # evaluate only at the end of each epoch
    save_strategy="epoch",               # save model at end of epoch
    report_to="none",                    # disables W&B logging
    fp16=True                            # use mixed precision if GPU available (faster)
)

In [ ]:
trainer = Trainer(model=model_bart, args=trainer_args,
                  tokenizer=tokenizer, data_collator=seq2seq_data_collator,
                  train_dataset=dataset_dailymail_pt["train"],
                  eval_dataset=dataset_dailymail_pt["validation"])

In [ ]:
trainer.train()

In [ ]:
import evaluate
from tqdm import tqdm
import torch

# Load metric
rouge = evaluate.load("rouge")


def generate_batch_sized_chunks(list_of_elements, batch_size):
    """Split the dataset into smaller batches"""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i: i + batch_size]


def calculate_metric_on_test_ds(dataset, model, tokenizer,
                                batch_size=16, device="cuda",
                                column_text="article",
                                column_summary="highlights"):

    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches),
            total=len(article_batches)):

        inputs = tokenizer(
            article_batch,
            max_length=1024,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        summaries = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            length_penalty=0.8,
            num_beams=8,
            max_length=128
        )

        # Decode predictions
        decoded_summaries = tokenizer.batch_decode(
            summaries,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        # Clean text
        decoded_summaries = [d.strip() for d in decoded_summaries]
        target_batch = [t.strip() for t in target_batch]

        # Add batch to metric
        rouge.add_batch(
            predictions=decoded_summaries,
            references=target_batch
        )

    # Compute final score
    score = rouge.compute()

    return score

In [ ]:
import evaluate

rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

rouge_metric = evaluate.load("rouge")
rouge_metric

In [ ]:
score = calculate_metric_on_test_ds(
    dataset_dailymail['test'][0:10],
    trainer.model,
    tokenizer,
    batch_size=2,
    column_text='article',
    column_summary='highlights'
)

# Extract scores directly
rouge_dict = {rn: score[rn] for rn in rouge_names}

import pandas as pd
df = pd.DataFrame(rouge_dict, index=['bart'])

print(df)

In [ ]:
## Save model
model_bart.save_pretrained("bart-dailymail-model")

In [ ]:
## Save tokenizer
tokenizer.save_pretrained("tokenizer")

In [ ]:
#Load
tokenizer = AutoTokenizer.from_pretrained("/content/tokenizer")

In [ ]:
#Prediction
gen_kwargs = {"length_penalty": 0.8, "num_beams":8, "max_length": 128}

In [ ]:
#Prediction

gen_kwargs = {"length_penalty": 0.8, "num_beams":8, "max_length": 128}



sample_text = dataset_dailymail["test"][0]["article"]

reference = dataset_dailymail["test"][0]["highlights"]

pipe = pipeline("summarization", model="bart-dailymail-model",tokenizer=tokenizer)

##
print("Article:")
print(sample_text)


print("\nReference highlights:")
print(reference)


print("\nModel highlights:")
print(pipe(sample_text, **gen_kwargs)[0]["summary_text"])